## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


When documents get long, they can have a lot of different information. Therefore to provide the right part of the document , we cut them up in smaller parts. This process is sometimes referred to as *chunking*.

In [ ]:
%pip install -q langchain markdown

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

We read the same document as a file first

In [ ]:
history_raw_text = ""
    # This is a long document we can split up.
with open("data/history.md") as f:
    history_raw_text = f.read()

When we use the generic splitter , it splits it per chunks . If useful we can make the chunks overlap too.

In [ ]:
# naive , generic chunksize splitter

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Set a really small chunk size, just to show.
text_splitter = RecursiveCharacterTextSplitter(
     chunk_size=100,
     chunk_overlap=0,
     length_function=len,
     add_start_index=True,
)
texts = text_splitter.create_documents([history_raw_text])

from pprint import pprint
pprint(texts)



But we can get smarter by using a content format aware splitter. In this case using Markdown header to do more meaningfull splits.

In [ ]:
# Now use a document/content specific splitter
from langchain.text_splitter import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
texts = md_splitter.split_text(history_raw_text)

pprint(texts)